# Q-Learning with FrozenLake


## 0. Install and Import Libraries
- `gymnasium`: provides the FrozenLake environment.
- `numpy`: stores and updates the Q-table.
- `matplotlib`: plots training results.

In [1]:
# If needed, uncomment the following line:
# !pip install gymnasium

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import sys
print(sys.executable)
print(sys.version)

## Introduction to Gymnasium
[Gymnasium](https://gymnasium.farama.org/) provides environments for reinforcement learning. In this lab, we use its **FrozenLake-v1** environment.

If `import gymnasium` fails in Google Colab, run `!pip install gymnasium` in a new cell and then retry the import.


In this lab, let's use Q-learning to train an agent to play FrozenLake.


## 1. Load FrozenLake Environment

We start with a 4×4 FrozenLake environment and **non-slippery** (deterministic) movement.
Later, change `IS_SLIPPERY = True` and retrain to explore stochastic transitions.

**Note:** The Python action IDs are `0=Left, 1=Down, 2=Right, 3=Up`. These IDs differ from the `a1/a2/a3/a4` labels on the lecture diagrams.


In [ ]:
IS_SLIPPERY = False

env = gym.make("FrozenLake-v1", is_slippery=IS_SLIPPERY)
num_states = env.observation_space.n
num_actions = env.action_space.n

action_names = {0: "Left", 1: "Down", 2: "Right", 3: "Up"}

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
for action_id, action_name in action_names.items():
    print(action_id, "=", action_name)


## 2. Set Hyperparameters

- `alpha = 0.2` controls the size of each Q-value update.
- `gamma = 0.95` determines the weight of estimated future rewards.
- `epsilon = 0.5` means a **50% chance of sampling a random action** and a 50% chance of choosing a current greedy action. A random action can coincidentally be greedy.
- `seed = 42` is used to make the demonstration reproducible.


In [ ]:
num_episodes = 5000
alpha = 0.2
gamma = 0.95
epsilon = 0.5
seed = 42

rng = np.random.default_rng(seed)
env.action_space.seed(seed)
print("Maximum steps per episode:", env.spec.max_episode_steps)


## 3. The Q-Learning Update Formula

For an **observed transition with a non-terminal next state**:

$$y = r + \gamma \max_{a'} Q(s', a')$$

For a **terminal next state**, $y=r$.

In both cases:

$$Q(s,a) \leftarrow Q(s,a)+\alpha\left[y-Q(s,a)\right]$$

The target uses an *observed reward* and an *estimated future value*; it is not necessarily the final true value.


## 4. Train the Agent

Initialize the Q-table to zeros. For each episode:

1. Reset the environment and choose each action using ε-greedy.
2. Observe the next state, reward, and termination/truncation flags.
3. Calculate the correct terminal/non-terminal target; update the Q-value for the **action actually taken**.
4. Repeat until termination or the 100-step limit; record training results.

**Important:** The training success rate includes exploratory actions. It is not the same as the performance of the learned greedy policy.


In [ ]:
# Initialize the Q-table and training metrics.
Q_values = np.zeros((num_states, num_actions))
episode_returns = []
episode_steps = []
success_rate = []

for iteration in range(num_episodes):
    state, info = env.reset(seed=seed if iteration == 0 else None)
    episode_return = 0

    for step in range(env.spec.max_episode_steps):
        # 1. Behavior: epsilon-greedy action selection
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q_values[state]))

        # 2. Observe an actual transition
        next_state, reward, terminated, truncated, info = env.step(action)

        # 3. Q-learning target: terminal states have no future return
        if terminated:
            target = reward
        else:
            target = reward + gamma * np.max(Q_values[next_state])

        # 4. Q-value update for the action that was actually taken
        Q_values[state, action] += alpha * (target - Q_values[state, action])
        state = next_state
        episode_return += reward

        if terminated or truncated:
            break

    episode_returns.append(episode_return)
    episode_steps.append(step + 1)
    success_rate.append(int(episode_return > 0))

    if (iteration + 1) % 1000 == 0:
        recent_success = np.mean(success_rate[-1000:]) * 100
        print(f"Episode {iteration+1} | Training success (last 1000): {recent_success:.1f}%")

print("Training finished. Episodes:", len(episode_returns))


## 5. Visualize Training Success

The curve below shows the **moving average of training success**, including exploratory moves. For the performance of the learned greedy policy, see the evaluation section.


In [ ]:
window = 100
training_success = np.asarray(success_rate, dtype=float)
moving_success = np.convolve(training_success, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 3.5))
plt.plot(np.arange(window, num_episodes + 1), moving_success)
plt.xlabel("Training episode")
plt.ylabel("Success rate (100-episode moving average)")
plt.title("FrozenLake — Q-Learning During Training")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.show()


## 6. Inspect the Learned Q-Table

Each row is a state and each column is a Gymnasium action (`0=Left`, `1=Down`, `2=Right`, `3=Up`).

In the policy visualization, `HOLE` and `GOAL` denote terminal squares; those are not states in which the agent should select another action.


In [ ]:
np.set_printoptions(precision=3, suppress=True)
print("Learned Q-table:")
print(Q_values)

best_actions = np.argmax(Q_values, axis=1)
best_action_names = np.array([action_names[a] for a in best_actions], dtype=object).reshape(4, 4)
map_tiles = env.unwrapped.desc.astype(str)
best_action_names[map_tiles == "H"] = "HOLE"
best_action_names[map_tiles == "G"] = "GOAL"
print("Greedy action per state (terminal states labeled):\n", best_action_names)


## 7. Evaluate the Learned Policy

During evaluation we choose the greedy action (`argmax Q`) without ε-greedy exploration.

Because a Colab notebook does not reliably support opening a desktop game window, this demonstration reports the state path and a success rate instead of using `render_mode="human"`.

**Important:** Testing must use the **same slippery setting** as training. In a deterministic lake, evaluations with different seeds may yield the same path; in a stochastic lake, multiple runs are especially useful.


In [ ]:
def run_greedy_episode(Q_values, test_seed=42):
    test_env = gym.make("FrozenLake-v1", is_slippery=IS_SLIPPERY)
    state, info = test_env.reset(seed=test_seed)
    path = [state]
    total_reward = 0

    for _ in range(test_env.spec.max_episode_steps):
        action = int(np.argmax(Q_values[state]))
        next_state, reward, terminated, truncated, info = test_env.step(action)
        total_reward += reward
        state = next_state
        path.append(state)
        if terminated or truncated:
            break

    test_env.close()
    return total_reward, path

# One example rollout
reward, path = run_greedy_episode(Q_values)
print("Greedy policy: reward =", reward)
print("Path as states:", path)
print("Path as (row, column):", [(s // 4, s % 4) for s in path])

# Evaluate over multiple episodes (different seeds are most useful if slippery=True)
evaluation_rewards = [run_greedy_episode(Q_values, test_seed=100+i)[0] for i in range(100)]
print("Greedy policy evaluation success rate:", f"{np.mean(evaluation_rewards)*100:.1f}%")


## Discussion Questions

1. What part of the training loop is exploration? What part is exploitation?
2. Why does the Q-learning target use the **maximum next-state Q-value**, even if we explored?
3. Why might the *training* success rate be lower than the *greedy evaluation* success rate?
4. Change `epsilon` (e.g., from 0.5 to 0.1). Predict, retrain from an initialized Q-table, and compare the learning curves.
5. Change `IS_SLIPPERY` to `True`, retrain, and re-evaluate. Does the learned policy or its stability change?
6. What happens if you change `alpha` or `gamma` while keeping other settings fixed?
7. What might make tabular Q-Learning impractical for image-based games?

When comparing configurations, **rerun the whole notebook from the beginning** to reinitialize the Q-table and keep training/evaluation environments consistent.
